# paper_riskDD — 01: Participant Screening

All-trials probit model (model-0, both formats, no group term) to identify task-disengaged participants.

**Exclusion criterion:** < 99.9 % of the posterior slope (γ) mass above zero — insufficient evidence that
the subject uses the numerical ratio when deciding.

Final agreed exclusion list (same for both formats): `[32, 40, 45, 46, 50]`

**Indifference point** = `−intercept / γ`  (log-ratio at which P(chose risky) = 0.5).
Risk-neutral value: `log(1/0.55) ≈ 0.598`.

**Outputs**
- `results/screening_per_subject_gamma.csv`
- `results/excluded_subjects.csv`
- `figures/screening_gamma_indpoint_by_group.pdf`
- `figures/screening_per_subject_gamma_kdeplots.pdf`

In [ ]:
import sys, os, os.path as op
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import bambi
import arviz as az

from numrisk.behavior_risk.utils import get_data, invprobit
sys.path.insert(0, os.getcwd())
from utils_riskBehav import summarize_posterior

sns.set_theme('paper', 'white', font='helvetica', font_scale=1.2)

# ind_point = -intercept / gamma  (log-ratio where P(chose risky) = 0.5)
# risk-neutral value: log(1/0.55) ≈ 0.598
IND_NEUTRAL = -np.log(0.55)

bids_folder    = '/Users/mrenke/data/ds-dnumrisk'
trace_folder   = op.join(bids_folder, 'derivatives', 'cogmodels_risk')
out_folder     = op.join(bids_folder, 'plots_and_ims', 'paper_riskDD')
figures_folder = op.join(out_folder, 'figures')
results_folder = op.join(out_folder, 'results')
for d in [figures_folder, results_folder]:
    os.makedirs(d, exist_ok=True)
print('Output folders ready.')

## Load behavioral data

In [ ]:
df = get_data()
df['x'] = df['log(risky/safe)']

subList   = df.index.unique('subject').sort_values()
groupList = df[['group']].groupby('subject').first().astype(int)
groupList['group_label'] = groupList['group'].map({0: 'Control', 1: 'Dyscalculic'})

print(f'Subjects loaded: {len(subList)}')
print(groupList['group_label'].value_counts())

---
## 1. Model-0: All-trials probit (no group term)

Formula: `chose_risky ~ x + x*format + (x*format|subject)`

Trace: `probit_model-0_format-both_trace.netcdf`

Model-0 has no group regressor — used purely to assess whether each subject's slope γ is
credibly different from zero.

In [ ]:
trace_m0 = az.from_netcdf(op.join(trace_folder, 'probit_model-0_format-both_trace.netcdf'))

model_m0 = bambi.Model(
    'chose_risky ~ x + x*format + (x*format|subject)',
    link='probit', family='bernoulli', data=df.reset_index()
)
print('Model-0 loaded.')
print('Posterior dims:', dict(trace_m0.posterior.dims))

### Extract per-subject slope γ and indifference point

Build a fake-data grid over `subject × x={0,1} × format` (no group term in model-0).

- **γ** = `invprobit(pred at x=1) − invprobit(pred at x=0)`
- **ind_point** = `−intercept / γ`  (log-ratio at which P=0.5; risk-neutral ≈ 0.598)

In [ ]:
fake_data_m0 = pd.MultiIndex.from_product(
    [subList, [0, 1], df.index.unique('format')],
    names=['subject', 'x', 'format']
).to_frame().reset_index(drop=True)

pred_m0 = model_m0.predict(
    trace_m0, 'mean', fake_data_m0, inplace=False, include_group_specific=True
)['posterior']['chose_risky_mean']

pred_m0 = pred_m0.to_dataframe().unstack([0, 1])
pred_m0 = pred_m0.set_index(pd.MultiIndex.from_frame(fake_data_m0))

pred0_m0     = pred_m0.xs(0, level='x')
intercept_m0 = pd.DataFrame(invprobit(pred0_m0), index=pred0_m0.index, columns=pred0_m0.columns)
gamma_m0     = invprobit(pred_m0.xs(1, level='x')) - intercept_m0

print('γ and intercept extracted.')
print('Index levels:', gamma_m0.index.names)

### Exclusion criterion: P(γ > 0) ≥ 0.999

In [ ]:
ALPHA = 0.001  # require 99.9 % of posterior mass above zero

excluded_m0 = []
records = []

for sub in subList:
    tmp_sub    = gamma_m0.xs(sub, level='subject').unstack('format')
    p_positive = float((tmp_sub.values > 0).mean())
    records.append({
        'subject':     sub,
        'group':       groupList.loc[sub, 'group'],
        'group_label': groupList.loc[sub, 'group_label'],
        'p_gamma_gt0': p_positive,
        'excluded':    p_positive < (1 - ALPHA)
    })
    if p_positive < (1 - ALPHA):
        excluded_m0.append(sub)

df_screening = pd.DataFrame(records).set_index('subject')

print('=== Subjects failing exclusion criterion (model-0) ===')
print(df_screening[df_screening['excluded']].to_string())
print(f'\nExcluded list (model-0): {excluded_m0}')

### Compute per-subject MAPs for γ and indifference point

In [ ]:
gamma_maps    = []
indpoint_maps = []

for sub in subList:
    g = gamma_m0.xs(sub, level='subject').values
    i = intercept_m0.xs(sub, level='subject').values
    gamma_maps.append(float(g.mean()))
    # ind_point = -intercept / gamma; averaged over posterior samples and formats
    with np.errstate(divide='ignore', invalid='ignore'):
        indpoint_maps.append(float((-i / g).mean()))

df_screening['gamma_map']    = gamma_maps
df_screening['indpoint_map'] = indpoint_maps

print(f'Risk-neutral ind_point = log(1/0.55) ≈ {IND_NEUTRAL:.3f}')
print()
print(df_screening[['group_label', 'gamma_map', 'indpoint_map', 'p_gamma_gt0', 'excluded']]
      .sort_values('gamma_map').to_string())

### Plot: γ and indifference point per subject

In [ ]:
pal = {'Control': sns.color_palette()[0], 'Dyscalculic': sns.color_palette()[1]}

fig, axs = plt.subplots(1, 2, figsize=(8, 3.5))

for ax, yvar, ylabel, ref_y in zip(
    axs,
    ['gamma_map', 'indpoint_map'],
    ['Slope γ (probit units)', 'Indifference point (log scale)'],
    [0,            IND_NEUTRAL]
):
    sns.swarmplot(
        data=df_screening.reset_index(),
        x='group_label', y=yvar, hue='group_label',
        palette=pal, ax=ax, size=5, legend=False
    )
    # mark excluded subjects with a red cross
    excl = df_screening[df_screening['excluded']].reset_index()
    groups_order = df_screening['group_label'].unique().tolist()
    for _, row in excl.iterrows():
        x_pos = groups_order.index(row['group_label'])
        ax.scatter(x_pos, row[yvar], marker='x', color='red', s=100, zorder=5)
    ax.axhline(ref_y, color='k', lw=0.8, ls='--')
    ax.set(xlabel='', ylabel=ylabel)
    sns.despine(ax=ax)

axs[1].set_ylabel('Indifference point  (= log 1/RNP)')
axs[1].annotate(f'risk-neutral\n({IND_NEUTRAL:.2f})', xy=(0.02, IND_NEUTRAL),
                xycoords=('axes fraction', 'data'), va='bottom', fontsize=8, color='k')
axs[0].legend(
    handles=[Line2D([0],[0], marker='x', color='red', ls='none', ms=8, label='Excluded')],
    frameon=False
)
plt.suptitle('Model-0: per-subject γ and indifference point', y=1.01)
plt.tight_layout()
fig.savefig(op.join(figures_folder, 'screening_gamma_indpoint_by_group.pdf'), bbox_inches='tight')
plt.show()
print('Figure saved.')

### Posterior γ KDE per subject — sorted by γ MAP

In [ ]:
gamma_stacked = gamma_m0.stack([1, 2])  # stack chain and draw into rows
gamma_stacked.columns = ['gamma']

sub_order = df_screening['gamma_map'].sort_values().index
n_sub = len(sub_order)
ncols = 8
nrows = int(np.ceil(n_sub / ncols))

fig, axs = plt.subplots(nrows, ncols, figsize=(ncols * 1.5, nrows * 1.5))
axs = axs.flatten()

for idx, sub in enumerate(sub_order):
    ax    = axs[idx]
    data  = gamma_stacked.xs(sub, level='subject')['gamma']
    grp   = groupList.loc[sub, 'group_label']
    excl  = df_screening.loc[sub, 'excluded']
    sns.kdeplot(data, ax=ax, fill=True, color=pal[grp], alpha=0.7)
    ax.axvline(0, color='k', lw=0.8, ls='--')
    ax.set(yticks=[], xlabel='', title=f'sub-{sub:02d}' + (' ✗' if excl else ''))
    ax.title.set_color('red' if excl else 'black')
    sns.despine(ax=ax, left=True)

for ax in axs[n_sub:]:
    ax.set_visible(False)

plt.suptitle('Posterior γ distributions — sorted by MAP (red = excluded)', y=1.01)
plt.tight_layout()
fig.savefig(op.join(figures_folder, 'screening_per_subject_gamma_kdeplots.pdf'), bbox_inches='tight')
plt.show()
print('Figure saved.')

---
## 2. Finalize exclusion list and save

Same 5 subjects excluded for **both** formats.

In [ ]:
EXCLUDED_SUBJECTS = [32, 40, 45, 46, 50]  # confirmed across model-0 and model-1; applied to both formats

df_screening['excluded_final'] = df_screening.index.isin(EXCLUDED_SUBJECTS)

df_screening.to_csv(op.join(results_folder, 'screening_per_subject_gamma.csv'))
pd.DataFrame({'subject': EXCLUDED_SUBJECTS}).to_csv(
    op.join(results_folder, 'excluded_subjects.csv'), index=False
)

print('=== EXCLUSION SUMMARY ===')
print(f'Total subjects:          {len(subList)}')
print(f'Excluded (model-0):      {sorted(excluded_m0)}')
print(f'Excluded (agreed final): {EXCLUDED_SUBJECTS}  [same for both formats]')
ctrl_excl = [s for s in EXCLUDED_SUBJECTS if groupList.loc[s,'group']==0]
dysc_excl = [s for s in EXCLUDED_SUBJECTS if groupList.loc[s,'group']==1]
print(f'  Controls excluded:     {ctrl_excl}')
print(f'  Dyscalculics excluded: {dysc_excl}')
n_incl = len(subList) - len(EXCLUDED_SUBJECTS)
n_ctrl = (groupList.loc[~groupList.index.isin(EXCLUDED_SUBJECTS),'group']==0).sum()
n_dysc = (groupList.loc[~groupList.index.isin(EXCLUDED_SUBJECTS),'group']==1).sum()
print(f'\nN for main analyses:     {n_incl}  (Controls: {n_ctrl}, Dyscalculics: {n_dysc})')

---
## 3. Basic characterization — included subjects

γ and indifference point distributions by group for the final sample (N=61, both formats pooled).

In [ ]:
df_included = df_screening[~df_screening['excluded_final']].copy()

fig, axs = plt.subplots(1, 2, figsize=(7, 3))

for ax, yvar, ylabel, ref_y in zip(
    axs,
    ['gamma_map', 'indpoint_map'],
    ['Slope γ (probit units)', 'Indifference point  (= log 1/RNP)'],
    [0,            IND_NEUTRAL]
):
    sns.boxplot(
        data=df_included.reset_index(), x='group_label', y=yvar,
        hue='group_label', palette=pal, ax=ax, width=0.4, fliersize=0, legend=False
    )
    sns.swarmplot(
        data=df_included.reset_index(), x='group_label', y=yvar,
        hue='group_label', palette=pal, ax=ax, size=4, alpha=0.6, legend=False
    )
    ax.axhline(ref_y, color='k', lw=0.8, ls='--')
    ax.set(xlabel='', ylabel=ylabel)
    sns.despine(ax=ax)

plt.suptitle('Basic characterization — included subjects (N=61)', y=1.01)
plt.tight_layout()
fig.savefig(op.join(figures_folder, 'characterization_gamma_indpoint_included.pdf'), bbox_inches='tight')
plt.show()

print('=== γ and indifference point by group (included subjects) ===')
print(df_included.groupby('group_label')[['gamma_map', 'indpoint_map']]
      .agg(['mean', 'std', 'median']).round(3).to_string())